## Лабораторная работа 6: Дискретное косинусное преобразование (DCT)

### Упражнения 6.2-6.3

In [ ]:
import sys
sys.path.insert(0, '../ThinkDSP/code')

from thinkdsp import read_wave
from thinkdsp import decorate
import matplotlib.pyplot as plt
import numpy as np

### Упражнение 6.2: Сжатие аудио с помощью DCT

Применим DCT для сжатия реального аудиосигнала.

In [ ]:
# Загрузка аудиофайла
wave = read_wave('../ThinkDSP/code/100475__iluppai__saxophone-weep.wav')

# Берем сегмент для анализа
start = 1.0
duration = 0.5
segment = wave.segment(start=start, duration=duration)
segment.normalize()

# Визуализация
segment.plot()
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

# Прослушивание оригинала
segment.make_audio()

Вычислим DCT сегмента:

In [ ]:
seg_dct = segment.make_dct()
seg_dct.plot(high=4000)
decorate(xlabel='Частота (Гц)', ylabel='DCT')
plt.show()

Функция для сжатия - обнуляет коэффициенты ниже порога:

In [ ]:
def compress(dct, thresh=1):
    """
    Обнуляет коэффициенты DCT ниже порога
    """
    count = 0
    for i, amp in enumerate(dct.amps):
        if np.abs(amp) < thresh:
            dct.hs[i] = 0
            count += 1
            
    n = len(dct.amps)
    print(f'Обнулено: {count} из {n} ({100 * count / n:.1f}%)')
    return count, n

Применим сжатие с порогом 10:

In [ ]:
seg_dct = segment.make_dct()
compress(seg_dct, thresh=10)
seg_dct.plot(high=4000)
decorate(xlabel='Частота (Гц)', ylabel='DCT (сжатый)')
plt.show()

Восстановим сигнал и прослушаем:

In [ ]:
seg2 = seg_dct.make_wave()
seg2.plot()
decorate(xlabel='Время (с)', ylabel='Амплитуда')
plt.show()

seg2.make_audio()

**Комментарий:** DCT позволяет эффективно сжимать аудио, так как большинство энергии сигнала сконцентрировано в небольшом числе коэффициентов. Обнуление малых коэффициентов практически не влияет на воспринимаемое качество звука.

### Упражнение 6.3: Сравнение разных порогов сжатия

In [ ]:
# Тестируем разные пороги
thresholds = [1, 5, 10, 20, 50]

fig, axes = plt.subplots(len(thresholds), 2, figsize=(12, 12))

for i, thresh in enumerate(thresholds):
    # Создаем новый DCT для каждого порога
    seg_dct = segment.make_dct()
    count, n = compress(seg_dct, thresh=thresh)
    
    # График DCT
    axes[i, 0].plot(seg_dct.fs[:2000], np.abs(seg_dct.hs[:2000]))
    axes[i, 0].set_title(f'DCT (порог={thresh}, обнулено {100*count/n:.1f}%)')
    axes[i, 0].set_xlabel('Частота (Гц)')
    axes[i, 0].set_ylabel('Амплитуда')
    
    # График восстановленного сигнала
    seg_restored = seg_dct.make_wave()
    axes[i, 1].plot(seg_restored.ts[:1000], seg_restored.ys[:1000])
    axes[i, 1].set_title(f'Восстановленный сигнал (порог={thresh})')
    axes[i, 1].set_xlabel('Время (с)')
    axes[i, 1].set_ylabel('Амплитуда')

plt.tight_layout()
plt.show()

Вычислим ошибку восстановления для разных порогов:

In [ ]:
thresholds = [0, 1, 5, 10, 20, 50, 100]
compression_ratios = []
errors = []

for thresh in thresholds:
    seg_dct = segment.make_dct()
    count, n = compress(seg_dct, thresh=thresh)
    
    # Степень сжатия
    compression_ratios.append(100 * count / n)
    
    # Ошибка восстановления (MSE)
    seg_restored = seg_dct.make_wave()
    mse = np.mean((segment.ys - seg_restored.ys)**2)
    errors.append(mse)
    
    print(f'Порог={thresh:3d}: сжатие={compression_ratios[-1]:5.1f}%, MSE={mse:.6f}')

In [ ]:
# График зависимости ошибки от степени сжатия
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(thresholds, compression_ratios, 'o-')
plt.xlabel('Порог')
plt.ylabel('Степень сжатия (%)')
plt.title('Зависимость сжатия от порога')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(compression_ratios, errors, 'o-')
plt.xlabel('Степень сжатия (%)')
plt.ylabel('MSE')
plt.title('Ошибка vs Сжатие')
plt.grid(True)

plt.tight_layout()
plt.show()

**Комментарий:** С увеличением порога растет степень сжатия, но также увеличивается ошибка восстановления. Оптимальный порог зависит от требуемого баланса между качеством и степенью сжатия.